In [1]:
from pyscf import gto, scf, cc

d = 100

o2 = '''
O 0.0 0.0 0.0
O 0.0 0.0 1.20577
'''

m_list = [1]
for nc in m_list:
    atoms = ""
    for n in range(nc):
        shift = n*d
        atoms += f'O {0.0+shift} 0.0 0.0     \n'
        atoms += f'O {0.0+shift} 0.0 1.20577 \n'

    # nfrozen = 2*nc
    spin = 2*nc
    mol = gto.M(atom=atoms, basis="sto6g", spin=spin, verbose=4)
    mol.build()

    mf = scf.UHF(mol)#.density_fit()
    mf.kernel()
    
    stable = False
    while not stable:
        print(f'mean-field stability test')
        if not stable:
            mo_i, _, stable,_ = mf.stability(return_status=True)
            dm = mf.make_rdm1(mo_i,mf.mo_occ)
            mf.kernel(dm0=dm)
        elif stable:
            print(f'UHF Energy: {mf.e_tot}, stability {stable}')
            break


    mycc = cc.CCSD(mf)
    mycc.set_frozen()
    mycc.kernel()

System: uname_result(system='Linux', node='yichi-thinkpad', release='4.4.0-26100-Microsoft', version='#8521-Microsoft Fri Jan 01 08:00:00 PST 2016', machine='x86_64')  Threads 12
Python 3.10.16 | packaged by conda-forge | (main, Dec  5 2024, 14:16:10) [GCC 13.3.0]
numpy 1.24.3  scipy 1.14.1  h5py 3.12.1
Date: Sun Jun 28 17:39:33 2026
PySCF version 2.12.1
PySCF path  /home/yichi/research/software/pyscf
GIT ORIG_HEAD a0665c4a7bf54e33f01295b3eea390be7a17d76d
GIT HEAD (branch master) f97393b29b0a541c155a68d55ee5b652ae7131d2

[ENV] OLD_PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:
[ENV] PYSCF_EXT_PATH /home/yichi/research/software/pyscf-forge:/home/sharmagroup/sharmagroup/pyscf-forge:
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 2
[INPUT] num. electrons = 16
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 2
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y                Z      unit      

In [2]:
import time
import numpy as np
# from jax import random
from jax import numpy as jnp

from afqmc import config
from afqmc import prep, sampling

from functools import partial
print = partial(print, flush=True)
config.setup_jax()

Hostname:     yichi-thinkpad
System:       Linux
Node:         yichi-thinkpad
Release:      4.4.0-26100-Microsoft
Machine:      x86_64
Processor:    x86_64
JAX backend:  CPU
JAX devices:  [CpuDevice(id=0)]
Device kind:  cpu
Platform:     cpu


In [3]:
from afqmc import integral
integral.prep_integral(mycc, chol_cut=1e-8)


Preparing AFQMC calculation
Calculating Cholesky integrals
Alpha Cholesky shape: (51, 8, 8) 
 Beta Cholesky shape: (51, 8, 8) 
Finished calculating Cholesky integrals
Size of the correlation space:
Number of electrons:        [7 5]
Number of basis functions:  8
Number of Cholesky vectors: 51


In [4]:
options = {'eql_time': 10,
           'n_blocks': 100,
           'n_walkers': 10,
           'mix_precision': False,
           'seed': 17,
           'guide': 'uhf',
           'trial': 'upt2ccsd',
           }

In [5]:
ham_data, ham, prop, trial, wave_data, sampler, options = (prep.init_afqmc(options=options))

wave_data["rdm1"] = trial.get_rdm1(wave_data)
ham_data = ham.build_measurement_intermediates(ham_data, trial, wave_data)
ham_data = ham.build_propagation_intermediates(ham_data, prop, trial, wave_data)
h0 = ham_data['h0']
prop_data = prep.init_hf_prop_data(trial, wave_data, ham_data, options)


Load system from Integral File
Maximum memory per walker:            200.00 MB
Maximum number of Cholesky per chunk: 102400
Number of Cholesky chunks:            1
Number of Cholesky per chunk:         51
Number of padding Cholesky:           0

QMC System
Number of electrons: (7, 5)
Spin Multiplicity:   2
Number of orbitals:  8
Number of Chol:      51

QMC Parameters
eql_time        -         10
n_blocks        -        100
n_walkers       -         10
mix_precision   -      False
seed            -         17
guide           -        uhf
trial           -   upt2ccsd
dt              -      0.005
n_exp_terms     -          6
n_prop_steps    -         50
walker_type     -        uhf
n_batch         -          1
max_error       -        0.0
nchol_chunk     -         51
max_memory      -       2000
free_projection -      False

Initalize QMC walkers by HF


In [6]:
init_e = prop_data["e_estimate"]
init_w = np.sum(prop_data["weights"])
init_time = time.time()

print("\nEquilibration")

print(f"{'inv_T':>5s}  "
      f"{'weight':>12s}  {'killW':>5s}  "
      f"{'energy':>12s}  {'runTime':>8s}")
print(f"{0.:5.2f}  "
      f"{init_w:12.5f}  {0:5d}  "
      f"{init_e:12.5f}  {time.time() - init_time:8.2f}")

sampler_eq = sampling.sampler(
    n_prop_steps=50, 
    n_chol = sampler.n_chol
    )

block_time = prop.dt * sampler_eq.n_prop_steps
neql_block = int(-(-options["eql_time"] // block_time))

for n in range(1, neql_block+1):
    prop_data, (wt, e) \
        = sampler_eq.block_sample(prop_data, ham_data, prop, trial, wave_data)

    if (n+1) % (min(max(neql_block // 10, 1), 20)) == 0 and n > 0:
        nkill = prop_data["n_killed_walkers"]
        print(f"{(n+1)*block_time:5.2f}  "
              f"{wt:12.5f}  {nkill:5d}  "
              f"{e:12.5f}  {time.time() - init_time:8.2f}")


Equilibration
inv_T        weight  killW        energy   runTime
 0.00      10.00000      0    -149.05334      0.00


 1.00      10.22541      0    -149.08861      3.10
 2.00      10.27778      0    -149.09106      3.19
 3.00      10.28875      0    -149.12985      3.26
 4.00      10.32650      0    -149.11712      3.33
 5.00      10.31152      0    -149.12289      3.40
 6.00      10.24372      0    -149.09690      3.49
 7.00      10.26726      0    -149.08162      3.57
 8.00      10.27539      0    -149.14296      3.66
 9.00      10.23408      0    -149.09760      3.75
10.00      10.24031      0    -149.15848      3.85


In [7]:
import jax
from jax import jit, lax
from jax import numpy as jnp
from jax import scipy as jsp
import opt_einsum as oe

In [8]:
norb = trial.norb
nocca, noccb = trial.nelec
t1a = wave_data["t1a"]
t1b = wave_data["t1b"]
t1a_full = np.zeros((norb, norb))
t1a_full[:nocca, nocca:] = t1a
t1b_full = np.zeros((norb, norb))
t1b_full[:noccb, noccb:] = t1b
wave_data['exp_t1a'] = jsp.linalg.expm(t1a_full)
wave_data['exp_mt1a'] = jsp.linalg.expm(-t1a_full)
wave_data['exp_t1b'] = jsp.linalg.expm(t1b_full)
wave_data['exp_mt1b'] = jsp.linalg.expm(-t1b_full)
chola = ham_data["chol"][0].reshape(-1, norb, norb)
cholb = ham_data["chol"][1].reshape(-1, norb, norb)
h1bar_a = wave_data['exp_t1a'] @ ham_data['h1'][0] @ wave_data['exp_mt1a']
h1bar_b = wave_data['exp_t1b'] @ ham_data['h1'][1] @ wave_data['exp_mt1b']
h1_bar = (h1bar_a, h1bar_b)
ham_data["h1_bar"] = h1_bar
chol_bar_a = oe.contract('pr,grs,sq->gpq', wave_data['exp_t1a'], chola, wave_data['exp_mt1a'], backend='jax')
chol_bar_b = oe.contract('pr,grs,sq->gpq', wave_data['exp_t1b'], cholb, wave_data['exp_mt1b'], backend='jax')
chol_bar = (chol_bar_a, chol_bar_b)
ham_data["chol_bar"] = chol_bar

In [9]:
(walker_up, walker_dn) = (prop_data["walkers"][0][0], prop_data["walkers"][1][0])
walker_up = jnp.eye(walker_up.shape[0])[:,:walker_up.shape[1]]
walker_dn = jnp.eye(walker_dn.shape[0])[:,:walker_dn.shape[1]]
t1, t2, e0, e1 = trial._calc_energy_pt(walker_up, walker_dn, ham_data, wave_data)
ept = (h0 + 1/t1*e0 + 1/t1*e1 - 1/t1**2 * t2 * e0).real
print(ept)
print(mycc.e_tot)

-149.16107384632883
-149.16107384656567


In [10]:
from afqmc import slater_tools, t2_tools
walker_up = wave_data["mo_coeff"][0] # prop_data["walkers"][0][0]
walker_dn = wave_data["mo_coeff"][1] # prop_data["walkers"][1][0]
walker_up = jnp.array(np.random.rand(*walker_up.shape)) + 1j*jnp.array(np.random.rand(*walker_up.shape))
walker_dn = jnp.array(np.random.rand(*walker_dn.shape)) + 1j*jnp.array(np.random.rand(*walker_dn.shape))
walker_up_bar = wave_data['exp_t1a'] @ walker_up
walker_dn_bar = wave_data['exp_t1b'] @ walker_dn
mo_t = (wave_data["mo_ta"], wave_data["mo_tb"])
ket = (walker_up, walker_dn)
h0 = ham_data["h0"]
h1 = ham_data["h1"]
chola = ham_data["chol"][0].reshape(-1,norb,norb)
cholb = ham_data["chol"][1].reshape(-1,norb,norb)
e0_tde = slater_tools.u_energy(mo_t,ket,h0,h1,(chola,cholb))
print(e0_tde)
mo = wave_data["mo_coeff"]
ket_bar = (walker_up_bar, walker_dn_bar)
e0_bar = slater_tools.u_energy(mo,ket_bar,h0,h1_bar,chol_bar)
print(e0_bar)

(-149.10961059907282-0.18182438893348274j)
(-149.10961059907282-0.18182438893348224j)


In [25]:
print(mf.e_tot+mycc.energy(mycc.t1,(mycc.t2[0]*0,mycc.t2[1]*0,mycc.t2[2]*0)))

-149.05815902803675


In [11]:
t2 = (wave_data["t2aa"], wave_data["t2ab"], wave_data["t2bb"])
olp_b, t2o_b, e012_b, t2e12_b = t2_tools.ut2h12_delta(mo, ket_bar, t2, h1_bar, chol_bar)
ept_b = h0 + e012_b + t2e12_b - t2o_b * e012_b
print(ept_b)
t1, t2, e0, e1 = trial._calc_energy_pt(walker_up, walker_dn, ham_data, wave_data)
ept_o = (h0 + 1/t1*e0 + 1/t1*e1 - 1/t1**2 * t2 * e0)
print(ept_o)

(-149.22060710538673+0.05551057317455843j)
(-149.2206071053868+0.055510573174551325j)


In [107]:
print(mf.e_tot+mycc.energy(mycc.t1,(mycc.t2[0]*0,mycc.t2[1]*0,mycc.t2[2]*0)))

-149.05815349684198


In [ ]:
from afqmc import walker_tools
walker_tools.map_over_walkers(single_fn, walkers, nbatch, *broadcast_args)